In [1]:
from google.cloud import aiplatform
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
PROJECT_ID = os.environ["GCP_PROJECT_ID"]
REGION = os.environ["GCP_REGION"]
aiplatform.init(project = PROJECT_ID, location=REGION)

In [3]:
serving_container_uri = "us-docker.pkg.dev/vertex-ai/prediction/pytorch-cpu.1-11:latest"

In [4]:
!gcloud storage buckets list --format="table(name, location, storage_url)"

NAME               LOCATION         STORAGE_URL
my-model-training  ASIA-SOUTHEAST1  gs://my-model-training/


In [5]:
!gcloud storage cp "model_serve/model.mar" "gs://my-model-training/result-mar/model.mar"

Copying file://model_serve/model.mar to gs://my-model-training/result-mar/model.mar
  Completed files 1/1 | 2.2kiB/2.2kiB                                          


In [6]:
!gcloud storage ls 'gs://my-model-training/result-mar'

gs://my-model-training/result-mar/model.mar


In [7]:
model = aiplatform.Model.upload(
    display_name="simple-model",
    serving_container_image_uri=serving_container_uri,
    artifact_uri='gs://my-model-training/result-mar',
)

Creating Model
Create Model backing LRO: projects/344969539300/locations/us-central1/models/5231503262993088512/operations/3414197205853011968
Model created. Resource name: projects/344969539300/locations/us-central1/models/5231503262993088512@1
To use this Model in another session:
model = aiplatform.Model('projects/344969539300/locations/us-central1/models/5231503262993088512@1')


In [8]:
!gcloud ai models list \
  --project=$PROJECT_ID \
  --region=$REGION \
  --format="table(name,displayName,createTime)"

Using endpoint [https://us-central1-aiplatform.googleapis.com/]
MODEL_ID             DISPLAY_NAME  CREATE_TIME
5231503262993088512  simple-model  2026-08-27T04:33:24.248534Z


In [9]:
endpoint = model.deploy(
    machine_type="n1-standard-4",
    accelerator_type=None,
    accelerator_count=0,
)

Creating Endpoint
Create Endpoint backing LRO: projects/344969539300/locations/us-central1/endpoints/7576994661504385024/operations/5247162254192803840
Endpoint created. Resource name: projects/344969539300/locations/us-central1/endpoints/7576994661504385024
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/344969539300/locations/us-central1/endpoints/7576994661504385024')
Deploying model to Endpoint : projects/344969539300/locations/us-central1/endpoints/7576994661504385024
Deploy Endpoint model backing LRO: projects/344969539300/locations/us-central1/endpoints/7576994661504385024/operations/1781642340931207168
Endpoint model deployed. Resource name: projects/344969539300/locations/us-central1/endpoints/7576994661504385024


In [13]:
response = endpoint.predict(instances=[
        {
            "body": {
                "data": [
                    [1.0, 2.0, 3.0, 4.0],
                    [1.0, 2.0, 3.0, 9.0]
                ]
            }
        }    
])

In [14]:
response

Prediction(predictions=[[[2.323722362518311], [3.395199298858643]]], deployed_model_id='4402880513975517184', metadata=None, model_version_id='1', model_resource_name='projects/344969539300/locations/us-central1/models/5231503262993088512', explanations=None)

In [15]:
endpoint.undeploy_all()
endpoint.delete()
model.delete()

Undeploying Endpoint model: projects/344969539300/locations/us-central1/endpoints/7576994661504385024
Undeploy Endpoint model backing LRO: projects/344969539300/locations/us-central1/endpoints/7576994661504385024/operations/6026408135030210560
Endpoint model undeployed. Resource name: projects/344969539300/locations/us-central1/endpoints/7576994661504385024
Deleting Endpoint : projects/344969539300/locations/us-central1/endpoints/7576994661504385024
Endpoint deleted. . Resource name: projects/344969539300/locations/us-central1/endpoints/7576994661504385024
Deleting Endpoint resource: projects/344969539300/locations/us-central1/endpoints/7576994661504385024
Delete Endpoint backing LRO: projects/344969539300/locations/us-central1/operations/2690876884450803712
Endpoint resource projects/344969539300/locations/us-central1/endpoints/7576994661504385024 deleted.
Deleting Model : projects/344969539300/locations/us-central1/models/5231503262993088512
Model deleted. . Resource name: projects/3